# iSWAP + OrientableArm: Simple Resource Move Test

Tests the `iSWAP` backend through `OrientableArm` with real PLR resources and the real STAR firmware interface.

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pylabrobot.arms.orientable_arm import OrientableArm
from pylabrobot.arms.standard import GripDirection
from pylabrobot.hamilton.liquid_handlers.star.iswap import iSWAP
from pylabrobot.legacy.liquid_handling.backends.hamilton.STAR_backend import STARBackend
from pylabrobot.resources.hamilton.hamilton_decks import STARLetDeck
from pylabrobot.resources.hamilton.plate_carriers import PLT_CAR_L5AC_A00
from pylabrobot.resources.corning.plates import Cor_96_wellplate_360ul_Fb

## Set up deck with carrier and plate

In [3]:
deck = STARLetDeck()

carrier = PLT_CAR_L5AC_A00("carrier")
deck.assign_child_resource(carrier, rails=14)

plate = Cor_96_wellplate_360ul_Fb("my_plate")
carrier[1].assign_child_resource(plate)

print(f"Plate '{plate.name}' is in: {plate.parent.name}")
print(f"Plate absolute location: {plate.get_absolute_location()}")
print(f"Destination site: {carrier[2].name}")
print(f"Destination location: {carrier[2].get_absolute_location()}")

Plate 'my_plate' is in: carrier-1
Plate absolute location: Coordinate(396.500, 167.500, 183.120)
Destination site: carrier-2
Destination location: Coordinate(396.500, 263.500, 186.150)


## Create iSWAP backend with real STAR interface

In [4]:
star_backend = STARBackend()
star_backend.set_deck(deck)
await star_backend.setup()

iswap_backend = iSWAP(interface=star_backend)

arm = OrientableArm(backend=iswap_backend, reference_resource=deck)
print(f"OrientableArm ready, iSWAP parked: {iswap_backend.parked}")

2026-03-20 22:03:16,770 - pylabrobot.io.usb - INFO - Finding USB device...
2026-03-20 22:03:16,811 - pylabrobot.io.usb - INFO - Found USB device.
2026-03-20 22:03:16,817 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0


cmd C0RMid0001
cmd C0QMid0002
cmd C0QWid0003
cmd C0ZAid0004
cmd C0EVid0005
cmd C0RTid0006
cmd I0QWid0007
cmd P1VYid0008
cmd P2VYid0009
cmd P3VYid0010
cmd P4VYid0011
cmd P5VYid0012
cmd P6VYid0013
cmd P7VYid0014
cmd P8VYid0015
cmd P9VYid0016
cmd PAVYid0017
cmd PBVYid0018
cmd PCVYid0019
cmd C0IVid0020
cmd R0QWid0021
cmd I0XPid0022xp54
cmd C0PGid0023th2800
cmd H0QWid0024
cmd H0RFid0025
cmd H0QUid0026
cmd H0QGid0027
OrientableArm ready, iSWAP parked: False


## Pick up plate from carrier[0]

In [10]:
await arm.pick_up_resource(
  plate,
  direction=GripDirection.FRONT,
  backend_params=iSWAP.PickUpParams(
    minimum_traverse_height=280.0,
    z_position_at_end=280.0,
    grip_strength=4,
    plate_width_tolerance=2.0,
    collision_control_level=0,
    fold_up_at_end=False,
  ),
)
print(f"Picked up '{plate.name}'")

cmd C0PPid0030xs04604xd0yj2102yd0zj1923zd0gr1th2800te2800gw4go1308gb1245gt20ga0gc0
Picked up 'my_plate'


In [9]:
await arm.park()

cmd C0PGid0029th2840


## Drop plate at carrier[2]

In [11]:
await arm.drop_resource(carrier[2], direction=GripDirection.FRONT)
print(f"Plate '{plate.name}' is now in: {plate.parent.name}")
print(f"Plate absolute location: {plate.get_absolute_location()}")

cmd C0PRid0031xs04604xd0yj3062yd0zj1923zd0th2800te2800gr1go1308ga0gc0
Plate 'my_plate' is now in: carrier-2
Plate absolute location: Coordinate(396.500, 263.500, 183.120)


In [13]:
await arm.pick_up_resource( plate, direction=GripDirection.LEFT,)
await arm.drop_resource(carrier[2], direction=GripDirection.RIGHT)

## Park and check state

In [ ]:
await arm.park()
print(f"iSWAP parked: {iswap_backend.parked}")